# Quantum Computing Laboratory

This master notebook contains the theoretical foundations, circuit layouts, and execution steps for 11 fundamental quantum computing experiments. All simulations are designed using Qiskit 1.x syntax and executed on the local Aer simulator.

## 1. Quantum Gates Playground

### Theory
Quantum gates are represented by unitary matrices. In this experiment, we explore:
*   **Pauli-X Gate:** Flips the state $|0\rangle$ to $|1\rangle$ (quantum NOT gate).
*   **Hadamard (H) Gate:** Creates a superposition state $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ by rotating vectors on the Bloch sphere.
*   **Pauli-Z Gate:** Applies a $\pi$ phase shift to $|1\rangle$ while leaving $|0\rangle$ unchanged.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Create 1-qubit circuit
qc = QuantumCircuit(1)
qc.h(0) # Apply H gate
qc.z(0) # Apply Z gate
print(qc.draw(output='text'))

# View statevector representation
sv = Statevector.from_instruction(qc)
print("Statevector:", sv.data)

## 2. Superposition (Single Qubit)

### Theory
Superposition allows a quantum system to exist in a linear combination of basis states. When measured, the state collapses to a basis state with probabilities determined by the amplitudes. For $|+\rangle$, the probability of measuring 0 or 1 is exactly 50%.

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

# 1-qubit circuit with measurement
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)
print(qc.draw(output='text'))

# Simulate execution
sim = AerSimulator()
job = sim.run(qc, shots=1024)
result = job.result()
counts = result.get_counts()
print("Measurement counts:", counts)

## 3. Bell State (Two Qubits)

### Theory
Bell states represent maximally entangled two-qubit states. We construct $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ using a Hadamard gate and a Controlled-NOT (CNOT) gate. Measurement will always yield correlated outcomes (`00` or `11`), demonstrating quantum correlation.

In [ ]:
# Create 2-qubit circuit
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1) # CNOT gate
qc.measure([0, 1], [0, 1])
print(qc.draw(output='text'))

# Run simulation
result = sim.run(qc, shots=1024).result()
print("Bell state counts:", result.get_counts())

## 4. Quantum Entanglement & GHZ State

### Theory
The Greenberger-Horne-Zeilinger (GHZ) state is a maximally entangled state of three or more qubits. For three qubits, it is defined as $|GHZ\rangle = \frac{1}{\sqrt{2}}(|000\rangle + |111\rangle)$. We construct this by cascading CNOT gates from the Bell state.

In [ ]:
# Create 3-qubit GHZ circuit
qc = QuantumCircuit(3, 3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)
qc.measure([0, 1, 2], [0, 1, 2])
print(qc.draw(output='text'))

# Run simulation
result = sim.run(qc, shots=1024).result()
print("GHZ state counts:", result.get_counts())

## 5. Quantum Circuits & Composition

### Theory
Quantum modularity involves composing separate registers and operations. Qiskit allows joining independent circuits together cleanly using the `.compose()` method.

In [ ]:
# Construct sub-circuits
qc_prep = QuantumCircuit(2)
qc_prep.h(0)
qc_prep.cx(0, 1)

qc_ops = QuantumCircuit(2)
qc_ops.x(0)
qc_ops.z(1)

# Compose and visualize
composed_qc = qc_prep.compose(qc_ops)
print(composed_qc.draw(output='text'))

## 6. Deutsch Algorithm

### Theory
The Deutsch algorithm determines if a one-bit function $f: \{0,1\} \rightarrow \{0,1\}$ is constant or balanced using only a single query. In a classical system, two queries are required. The input qubit will measure as $|1\rangle$ if the function is balanced, and $|0\rangle$ if it is constant.

In [ ]:
# Balanced oracle: f(x) = x (implemented using CNOT)
oracle = QuantumCircuit(2)
oracle.cx(0, 1)

# Deutsch circuit
qc = QuantumCircuit(2, 1)
qc.x(1)       # Set target to |1>
qc.h([0, 1])  # Put both in superposition
qc.barrier()
qc.compose(oracle, inplace=True) # Apply oracle
qc.barrier()
qc.h(0)       # Hadamard on input qubit
qc.measure(0, 0)
print(qc.draw(output='text'))

# Simulate. A balanced oracle should return '1'
result = sim.run(qc, shots=1024).result()
print("Result (1 = Balanced, 0 = Constant):", result.get_counts())

## 7. Grover Search Simulation

### Theory
Grover's algorithm searches unstructured databases of size $N$ in $O(\sqrt{N})$ queries, providing a quadratic speedup over classical $O(N)$ searches. Here we simulate a 2-qubit unstructured search for the target state $|11\rangle$. We implement the initialization, an oracle marking $|11\rangle$ by flipping its phase, and a diffusion operator to amplify the marked state's amplitude.

In [ ]:
# Grover's Search for target state |11>
qc = QuantumCircuit(2, 2)

# 1. Superposition state
qc.h([0, 1])
qc.barrier()

# 2. Oracle (CZ gate marks |11> by flipping its phase)
qc.cz(0, 1)
qc.barrier()

# 3. Diffusion Operator (H, Z, CZ, H cascade)
qc.h([0, 1])
qc.z([0, 1])
qc.cz(0, 1)
qc.h([0, 1])
qc.barrier()

# 4. Measurement
qc.measure([0, 1], [0, 1])
print(qc.draw(output='text'))

# Run simulation. State '11' should be measured with 100% probability
result = sim.run(qc, shots=1024).result()
print("Grover output counts:", result.get_counts())

## 8. Quantum Fourier Transform (QFT)

### Theory
The Quantum Fourier Transform (QFT) is the quantum analogue of the discrete Fourier transform. It maps computational state vectors to phase state vectors, acting as a core subroutine in Shor's algorithm. We implement a 3-qubit QFT using Hadamard and Controlled-Phase ($CP$) rotations.

In [ ]:
import numpy as np

qc = QuantumCircuit(3)

# Apply H and CP rotations on Qubit 2
qc.h(2)
qc.cp(np.pi/2, 1, 2)
qc.cp(np.pi/4, 0, 2)

# Apply H and CP on Qubit 1
qc.h(1)
qc.cp(np.pi/2, 0, 1)

# Apply H on Qubit 0
qc.h(0)

# Swap endpoints to match mathematical ordering
qc.swap(0, 2)
print(qc.draw(output='text'))

## 9. Shor Algorithm Simulation

### Theory
Shor's algorithm factors integers in polynomial time. The quantum component consists of finding the period $r$ of $a^x \pmod N$. Here we show a simplified 3-qubit phase estimation circuit representing period finding for $a=7, N=15$ (which has a period $r=4$).

In [ ]:
from qiskit import QuantumRegister, ClassicalRegister

# Define Registers
c_reg = QuantumRegister(3, name='control')
t_reg = QuantumRegister(4, name='target')
meas = ClassicalRegister(3, name='meas')
qc = QuantumCircuit(c_reg, t_reg, meas)

# Initialize target register to |1>
qc.x(t_reg[0])

# Put control register in superposition
qc.h(c_reg)

# Controlled multiplications (simplified modular exponentiation gates)
qc.ccx(c_reg[0], t_reg[0], t_reg[2])
qc.ccx(c_reg[0], t_reg[1], t_reg[3])

# Inverse QFT on control register
qc.swap(c_reg[0], c_reg[2])
qc.h(c_reg[0])
qc.cp(-np.pi/2, c_reg[0], c_reg[1])
qc.h(c_reg[1])
qc.cp(-np.pi/4, c_reg[0], c_reg[2])
qc.cp(-np.pi/2, c_reg[1], c_reg[2])
qc.h(c_reg[2])

# Measure control register
qc.measure(c_reg, meas)
print(qc.draw(output='text'))

# Run simulation
result = sim.run(qc, shots=1024).result()
print("Measurement outcomes:", result.get_counts())

## 10. Quantum Measurement

### Theory
Quantum measurement collapses a superposition state into one of the basis states. If we prepare the state $|+\rangle$ and measure in the Z basis (computational basis), we get `0` or `1` with equal probabilities. If we measure in the X basis (by applying a Hadamard gate prior to measurement), we get `0` with 100% probability because $|+\rangle$ matches the X-basis state.

In [ ]:
# Prepare superposition state |+>
qc = QuantumCircuit(1, 2)
qc.h(0)

# Measure in computational Z basis
qc.measure(0, 0)

# Measure in X basis (H gate shifts to X basis)
qc.h(0)
qc.measure(0, 1)
print(qc.draw(output='text'))

# Run simulation
result = sim.run(qc, shots=1024).result()
print("Measurements (right bit: Z basis, left bit: X basis):", result.get_counts())

## 11. Noise Simulation

### Theory
Noisy simulation models interactions of qubits with the environment, causing loss of quantum information (decoherence). We simulate **phase damping**, which models phase decay (T2 relaxation) in physical processors without changing energy states.

In [ ]:
from qiskit_aer.noise import NoiseModel, phase_damping_error

# Create phase damping error (15% phase flip probability)
error = phase_damping_error(0.15)
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(error, ['h'])

# Run Bell state circuit under noise
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

# Simulate with and without noise
clean_sim = AerSimulator()
noisy_sim = AerSimulator(noise_model=noise_model)

clean_counts = clean_sim.run(qc, shots=1024).result().get_counts()
noisy_counts = noisy_sim.run(qc, shots=1024).result().get_counts()

print("Ideal Bell counts:", clean_counts)
print("Noisy Bell counts:", noisy_counts)

### Learning Summary

### Data Analysis Key Findings
*   **Qubits and Gates:** Pauli gates manipulate single qubits, while the Hadamard gate establishes a uniform superposition state.
*   **Entanglement:** Bell states and GHZ states showcase maximal entanglement, yielding correlated measurement distributions.
*   **Algorithmic Acceleration:** The Deutsch and Grover algorithms confirm that quantum phase manipulation and amplitude amplification offer speedups over classical computing models.
*   **Environmental Noise:** Introducing phase damping decoherence splits state probability density, proving why error mitigation is vital for physical hardware operations.

### Insights or Next Steps
*   Run composed circuits on real IBM Quantum hardware nodes using the `qiskit-ibm-runtime` API provider.
*   Investigate error correction models (such as three-qubit bit flip codes) to counter environmental phase damping.